# Stock Price Target Prediction — ML Analytics Platform

**Version 2.0.0** — Production-Ready Streamlined Workflow

## Business Objective

**Primary Goal**: Predict Stock Price Targets for all stocks in the portfolio to support 
investment decisions and portfolio optimization.

**Target Variable**: "Predicted Price Target" for regression modeling

## Workflow Overview (10 Steps)

1. **Configuration and Setup** — Initialize environment and configuration
2. **Loading and Preprocessing** — Multi-region data with 4-step imputation
3. **Exploratory Data Analysis** — Financial metrics and benchmarking
4. **Feature Engineering** — Sector-specific optimizations
5. **Multi-Class Classification** — Financial event detection
6. **Sector-Optimized Regression** — Price target prediction with classification features
7. **Model Evaluation** — Comprehensive error analysis
8. **Stock Valuation** — Under/overvalued identification
9. **Predicted vs. Analyst Analytics** — Target comparison
10. **Portfolio Optimization** — Risk-adjusted portfolio construction

## Key Features

- 📊 **Data Management**: PostgreSQL/CSV with validation (data.py, data_catalog.py)
- 🔧 **Preprocessing**: 4-step imputation strategy (advanced_preprocessing.py)
- 📈 **EDA**: Statistical tests, benchmarking (advanced_eda.py, benchmarking.py, eval.py)
- 🔨 **Features**: Financial ratios, sector-specific (features.py, advanced_features.py, transformers.py)
- 🤖 **Models**: Classification + regression (classification.py, models.py, advanced_models.py)
- 📊 **Analytics**: Comprehensive evaluation (eval.py, analyst_comparison.py)
- 💼 **Portfolio**: Optimization with risk metrics (portfolio_optimization.py, risk_metrics.py)


## 1. Configuration and Setup


In [ ]:
# Import configuration
from finance_ml import NotebookConfig

# Initialize with production settings
config = NotebookConfig(
        have_finance_prediction=True,
        have_database_connection=True,
        have_advanced_analytics=True,
        have_dim_reduction=True,
        debug_mode=False,
        enable_sector_analysis=True,
        enable_region_analysis=True,
        enable_interactive_plots=True,
        enable_excel_export=True,
        )
config.display_summary()


In [ ]:
# Core imports
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Finance ML package imports
from finance_ml import (
    data, features, advanced_features, classification,
    advanced_models, eval as fm_eval, analyst_comparison,
    portfolio_optimization, risk_metrics
    )

# Specific function imports
from finance_ml.data import load_from_csv, load_from_db, validate_schema
from finance_ml.advanced_preprocessing import apply_enhanced_imputation_strategy_4step
from finance_ml.advanced_eda import generate_eda_report
from finance_ml.benchmarking import generate_benchmarking_report

warnings.filterwarnings('ignore')

# Set random seed
RANDOM_SEED = int(os.getenv('RANDOM_SEED', '42'))
np.random.seed(RANDOM_SEED)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Output directories
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(exist_ok=True)
(OUTPUT_DIR / "reports").mkdir(exist_ok=True)

print("✓ Configuration and imports complete")


## 2. Loading and Preprocessing Financial Data

Sophisticated preprocessing pipeline with:
1. **Data Loading**: Multi-region data from PostgreSQL or CSV
2. **Outlier Detection**: IQR, Z-score, and Isolation Forest methods
3. **Sector-Specific Winsorization**: Limit extreme values by sector
4. **Data Quality Scoring**: Comprehensive quality metrics
5. **4-Step Imputation Strategy**:
   - Zero imputation for metrics that can be zero
   - Price-based imputation for price-derived metrics
   - KNN imputation (sector-aware) for complex relationships
   - Median imputation (sector-aware) as final fallback
6. **Feature Scaling**: Robust scaling by sector


In [ ]:
# Load data (auto-detect from DB or CSV)
from finance_ml.data import load_from_db, load_from_csv, validate_schema
from finance_ml.data_catalog import DataCatalog

DB_URL = os.getenv('DB_URL', 'postgresql+psycopg2://postgres:@localhost:5432/postgres')

try:
    all_stocks = load_from_db(DB_URL, limit=None)
    print(f"✓ Loaded {len(all_stocks)} stocks from database")
except Exception as e:
    print(f"⚠ Database load failed: {e}. Falling back to CSV.")
    all_stocks = load_from_csv(Path("data"), limit=None)
    print(f"✓ Loaded {len(all_stocks)} stocks from CSV")

# Validate schema
validate_schema(all_stocks, require_target=True)

print(f"✓ Initial data shape: {all_stocks.shape}")
print(f"  Initial missing values: {all_stocks.isnull().sum().sum()}")


In [ ]:
# Robust outlier detection with multiple methods
from finance_ml.advanced_preprocessing import (
    detect_outliers_zscore,
    detect_outliers_isolation_forest,
    detect_outliers_iqr
    )

# Outlier Detection Section
print("\n" + "=" * 80)
print("OUTLIER DETECTION")
print("=" * 80)

# Detect outliers using multiple methods
numeric_cols = all_stocks.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = all_stocks.select_dtypes(include=[np.number]).columns.tolist()
financial_metrics = [c for c in numeric_cols if c not in ['ticker', 'isin']]

# Detect outliers using multiple methods
outliers_iqr = detect_outliers_iqr(
        all_stocks,
        columns=financial_metrics[:20],
        iqr_multiplier=1.5
        )

outliers_zscore = detect_outliers_zscore(
        all_stocks,
        columns=financial_metrics[:20],
        threshold=3.0
        )

outliers_iforest = detect_outliers_isolation_forest(
        all_stocks,
        columns=financial_metrics[:20],
        contamination=0.1,
        random_state=42
        )

# Ensure all outputs are boolean type
outliers_iqr = outliers_iqr.astype(bool)
outliers_zscore = outliers_zscore.astype(bool)
outliers_iforest = pd.Series(outliers_iforest).astype(bool) if isinstance(outliers_iforest, (list,
                                                                                             np.ndarray)) else outliers_iforest.astype(
    bool)

print(f"✓ Outliers detected:")
print(f"  IQR method: {outliers_iqr.sum().sum()} outliers")
print(f"  Z-score method: {outliers_zscore.sum().sum()} outliers")
print(f"  Isolation Forest: {outliers_iforest.sum()} outliers")


In [ ]:
# Sector-specific winsorization to handle extreme values
from finance_ml.advanced_preprocessing import winsorize_by_sector

print("\n✂️ Applying Sector-Specific Winsorization...")

# Winsorize key financial metrics by sector
all_stocks = winsorize_by_sector(
        all_stocks,
        columns=financial_metrics[:20],
        lower_percentile=0.01,
        upper_percentile=0.99,
        by_sector=True
        )

print(f"✓ Winsorization complete")
print(f"  Applied to {len(financial_metrics[:20])} financial metrics")


In [ ]:
# Calculate comprehensive data quality score
from finance_ml.advanced_preprocessing import calculate_data_quality_score

print("\n📊 Calculating Data Quality Scores...")

quality_report = calculate_data_quality_score(all_stocks)

print(f"✓ Data Quality Report:")
print(f"  Overall score: {quality_report.overall_score:.2f}")
print(f"  Completeness: {quality_report.completeness_score:.2f}")
print(f"  Validity: {quality_report.validity_score:.2f}")
print(f"  Consistency: {quality_report.consistency_score:.2f}")
print(f"  Issues detected: {len(quality_report.issues)}")

# Optionally, print the actual issues
if quality_report.issues:
    print(f"  Issue details:")
    for issue in quality_report.issues[:5]:  # Show first 5 issues
        print(f"    - {issue}")
    if len(quality_report.issues) > 5:
        print(f"    ... and {len(quality_report.issues) - 5} more issues")


In [ ]:
# Apply enhanced 4-step imputation strategy
from finance_ml.advanced_preprocessing import apply_enhanced_imputation_strategy_4step

print("\n📊 Applying Enhanced 4-Step Imputation Strategy...")
all_stocks = apply_enhanced_imputation_strategy_4step(
        all_stocks,
        sector_column='sector',
        n_neighbors=5,
        price_column='last_price'
        )

print(f"✓ Imputation complete")
print(f"  Missing values remaining: {all_stocks.isnull().sum().sum()}")


In [ ]:
# Apply feature scaling with robust scaler (by sector)
from finance_ml.advanced_preprocessing import scale_features

print("\n⚖️ Applying Feature Scaling...")

# Scale numeric features (excluding targets and identifiers)
exclude_scaling = ['ticker', 'isin', 'price_target', 'price_target_median', 'last_price']
scaling_cols = [c for c in numeric_cols if c not in exclude_scaling]

all_stocks_scaled = scale_features(
        all_stocks.copy(),
        columns=scaling_cols[:30],  # Scale key features
        scaler_type='robust',
        by_sector=True
        )

# Keep original data for regression, use scaled for classification
print(f"✓ Feature scaling complete")
print(f"  Scaled {len(scaling_cols[:30])} features using robust scaler")


In [ ]:
# Preprocessing summary
print("\n" + "=" * 80)
print("PREPROCESSING COMPLETE - Summary")
print("=" * 80)
print(f"✓ Final data shape: {all_stocks.shape}")
print(f"✓ Missing values: {all_stocks.isnull().sum().sum()}")
print(f"✓ Data quality score: {quality_report.overall_score:.2f}")
print(f"✓ Outlier detection: 3 methods applied")
print(f"✓ Winsorization: Sector-specific applied")
print(f"✓ Imputation: 4-step strategy applied")
print(f"✓ Feature scaling: Robust scaler by sector")
print("=" * 80)


## 3. Exploratory Data Analysis of Financial Metrics

Comprehensive statistical analysis including:
- Distribution analysis, outlier detection, normality tests
- Correlation matrices (Pearson, Spearman, Kendall)
- Sector and region comparisons with hypothesis tests
- Benchmarking and peer analysis


In [ ]:
# Generate comprehensive EDA report
from finance_ml.advanced_eda import generate_eda_report

eda_output_dir = OUTPUT_DIR / "eda"
eda_output_dir.mkdir(exist_ok=True)

eda_report = generate_eda_report(
        all_stocks,
        target_col='price_target',
        sector_col='sector',
        output_dir=eda_output_dir
        )

print(f"✓ EDA Report Generated")
print(
    f"  Correlations: {len(eda_report.correlation_analysis.pearson_matrix.columns) if eda_report.correlation_analysis else 0} features")
print(f"  Statistical tests: {len(eda_report.normality_tests)} performed")


In [ ]:
# Benchmarking analysis
from finance_ml.benchmarking import generate_benchmarking_report

metrics_to_benchmark = ['p_e', 'p_b', 'ev_ebitda', 'operating_margin', 'roe']
available_metrics = [m for m in metrics_to_benchmark if m in all_stocks.columns]

benchmark_report = generate_benchmarking_report(
        all_stocks,
        metrics=available_metrics,
        sector_column='sector',
        region_column='region'
        )

print(f"✓ Benchmarking Report Generated")
print(f"  Sectors analyzed: {benchmark_report['summary']['n_sectors']}")
print(f"  Regions analyzed: {benchmark_report['summary']['n_regions']}")


In [ ]:
# Key visualizations
from finance_ml.eval import simple_eda

simple_eda(
        all_stocks,
        out_dir=eda_output_dir,
        save_plots=True,
        target_column='price_target',
        include_multivariate=True
        )


## 4. Advanced Feature Engineering with Sector-Specific Optimizations

Features include:
- Financial ratios (valuation, profitability, leverage, liquidity, efficiency)
- Sector-specific features (Financials, Energy, Tech, Healthcare, etc.)
- Growth metrics and temporal features
- Relative value features (sector-normalized)
- Feature importance analysis


In [ ]:
# Build comprehensive features
from finance_ml.advanced_features import build_comprehensive_features

all_stocks_features = build_comprehensive_features(
        all_stocks,
        include_interactions=True,
        include_relative_values=True,
        sector_col='sector'
        )

print(f"✓ Feature Engineering Complete")
print(f"  Original features: {all_stocks.shape[1]}")
print(f"  Engineered features: {all_stocks_features.shape[1]}")
print(f"  New features added: {all_stocks_features.shape[1] - all_stocks.shape[1]}")


In [ ]:
# Feature importance analysis
from finance_ml.advanced_features import calculate_feature_importance_rf

exclude_cols = ['ticker', 'sector', 'region', 'price_target', 'last_price']
feature_cols = [c for c in all_stocks_features.columns if c not in exclude_cols]

if 'price_target' in all_stocks_features.columns:
    X = all_stocks_features[feature_cols].select_dtypes(include=[np.number])
    y = all_stocks_features['price_target']

    importance_df = calculate_feature_importance_rf(X, y, top_k=20)
    print("\n🎯 Top 20 Most Important Features:")
    print(importance_df)


## 5. Multi-Class Classification of Financial Events

Train sophisticated classification models to predict financial events:
- Event labeling: Neutral, Positive, Negative (price momentum method)
- Multiple classifiers: XGBoost, LightGBM, CatBoost, Neural Networks, Ensembles
- Export classification probabilities as meta-features for regression


In [ ]:
# Create event labels
from finance_ml.classification import create_enhanced_event_labels, prepare_classification_data

labels = create_enhanced_event_labels(
        all_stocks_features,
        method='price_momentum',
        threshold_positive=10.0,
        threshold_negative=-10.0,
        use_sector_adjustment=True
        )

print(f"✓ Event Labels Created")
print(f"  Class distribution:")
print(f"    Neutral (0): {(labels == 0).sum()} ({(labels == 0).sum() / len(labels) * 100:.1f}%)")
print(f"    Positive (1): {(labels == 1).sum()} ({(labels == 1).sum() / len(labels) * 100:.1f}%)")
print(f"    Negative (2): {(labels == 2).sum()} ({(labels == 2).sum() / len(labels) * 100:.1f}%)")


In [ ]:
# Compare multiple classifiers
from finance_ml.classification import compare_classifiers

# prepare_classification_data returns 6 values including numeric_cols and categorical_cols
X_train, X_test, y_train, y_test, numeric_cols, categorical_cols = prepare_classification_data(
        all_stocks_features, labels, test_size=0.2, random_state=RANDOM_SEED
        )

comparison_results = compare_classifiers(
        X_train, y_train, X_test, y_test,
        numeric_cols, categorical_cols
        )

print("\n📊 Classifier Comparison Results:")
print(comparison_results.sort_values('F1-Score', ascending=False))


In [ ]:
# Export classification features
from finance_ml.classification import train_stacking_classifier, export_classification_features

best_model = train_stacking_classifier(
        X_train, y_train, X_test, y_test,
        numeric_cols, categorical_cols
        )

# Get predictions for all data
X_all = pd.concat([X_train, X_test], axis=0)
y_proba_all = best_model['model'].predict_proba(X_all)

all_stocks_with_classification = export_classification_features(
        all_stocks_features,
        y_proba_all,
        class_names=["Neutral", "Positive", "Negative"]
        )

print(f"✓ Classification meta-features added: {all_stocks_with_classification.shape}")


## 6. Phase 9.5 — Sector-Optimized Regression Models with Classification Features

Advanced regression modeling using functions from `finance_ml.advanced_models`:

**Workflow Steps:**
1. Create interaction features between classification probabilities and valuation metrics
2. Prepare regression data with classification meta-features
3. Train and compare multiple regression models (Ridge, Lasso, RF, ET, GB, HistGB)
4. Build stacking ensemble for best performance
5. Train quantile regression for prediction intervals
6. Train sector-specific models (optional)
7. Save models with metadata
8. Store predictions for downstream analysis

**Key Functions:**
- `create_classification_interactions` — Create feature interactions
- `prepare_regression_data` — Split and preprocess data
- `compare_regressors` — Compare 6 regression models
- `train_stacking_regressor` — Build ensemble
- `train_quantile_regressor` — Prediction intervals
- `train_sector_specific_models` — Per-sector optimization
- `save_model` — Model persistence


In [ ]:
# Import required functions from finance_ml.advanced_models
from finance_ml.advanced_models import (
    prepare_regression_data,
    create_classification_interactions,
    compare_regressors,
    train_stacking_regressor,
    train_quantile_regressor,
    train_sector_specific_models,
    save_model
    )
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from datetime import datetime

# Configuration constants
TARGET_COL = 'price_target'
TARGET_COL_FALLBACK = 'last_price'
TEST_SIZE = 0.2
CV_FOLDS = 5
QUANTILES = [0.1, 0.5, 0.9]
MIN_SECTOR_SAMPLES = 20

print("✓ Phase 9.5 imports complete")


### 6.1 Create Classification Interaction Features


In [ ]:
print("=" * 80)
print("6.1 — Creating Classification Interaction Features")
print("=" * 80)

# Extract classification and valuation columns
classification_cols = [c for c in all_stocks_with_classification.columns if c.startswith('event_prob_')]
valuation_cols = [c for c in ['p_e', 'p_b', 'ev_ebitda', 'market_cap']
                  if c in all_stocks_with_classification.columns]

if classification_cols and valuation_cols:
    print(f"\nClassification features: {len(classification_cols)}")
    print(f"Valuation features: {len(valuation_cols)}")

    # Create interaction features
    all_stocks_enhanced = create_classification_interactions(
            all_stocks_with_classification,
            classification_cols=classification_cols,
            valuation_cols=valuation_cols
            )

    # Report results
    interaction_cols = [c for c in all_stocks_enhanced.columns
                        if '_x_' in c and c not in all_stocks_with_classification.columns]
    print(f"\n✓ Created {len(interaction_cols)} interaction features")
    if interaction_cols[:3]:
        print(f"  Examples: {', '.join(interaction_cols[:3])}")
else:
    print("\n⚠ Skipping interaction features - missing required columns")
    all_stocks_enhanced = all_stocks_with_classification.copy()

# Handle missing values
print("\n🔧 Handling missing values...")
nan_before = all_stocks_enhanced.isnull().sum().sum()
if nan_before > 0:
    all_stocks_enhanced = all_stocks_enhanced.replace([np.inf, -np.inf], np.nan)
    numeric_cols = all_stocks_enhanced.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col != TARGET_COL and all_stocks_enhanced[col].isnull().any():
            median_val = all_stocks_enhanced[col].median()
            all_stocks_enhanced[col] = all_stocks_enhanced[col].fillna(median_val if pd.notna(median_val) else 0)
    all_stocks_enhanced = all_stocks_enhanced.fillna(0)
    print(f"✓ Cleaned {nan_before} NaN values")


### 6.2 Prepare Regression Data


In [ ]:
print("=" * 80)
print("6.2 — Preparing Regression Data")
print("=" * 80)

# Use fallback target if needed
target_col = TARGET_COL if TARGET_COL in all_stocks_enhanced.columns else TARGET_COL_FALLBACK
if target_col == TARGET_COL_FALLBACK:
    print(f"⚠ Using '{TARGET_COL_FALLBACK}' as target ('{TARGET_COL}' not found)")

# Prepare train/test split
X_train_reg, X_test_reg, y_train_reg, y_test_reg, feature_info = prepare_regression_data(
        all_stocks_enhanced,
        target_col=target_col,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED
        )

print(f"\n✓ Data prepared:")
print(f"  Train set: {X_train_reg.shape}")
print(f"  Test set: {X_test_reg.shape}")
print(f"  Numeric features: {len(feature_info.get('numeric_features', []))}")
print(f"  Categorical features: {len(feature_info.get('categorical_features', []))}")


### 6.3 Compare Multiple Regression Models


In [ ]:
print("=" * 80)
print("6.3 — Comparing Multiple Regression Models")
print("=" * 80)

try:
    comparison_results = compare_regressors(
            X_train_reg, y_train_reg,
            test_size=TEST_SIZE,
            cv=CV_FOLDS,
            random_state=RANDOM_SEED,
            ensure_nonnegative=True
            )

    results_df = pd.DataFrame(comparison_results).T.sort_values('r2', ascending=False)
    print("\n📊 Model Comparison Results:")
    print(results_df.to_string())

    if not results_df.empty:
        best_model_name = results_df.index[0]
        print(f"\n🏆 Best Model: {best_model_name}")
        print(f"   R²: {results_df.loc[best_model_name, 'r2']:.4f}")
        print(f"   MAE: {results_df.loc[best_model_name, 'mae']:.2f}")
    else:
        best_model_name = "None"
        print("  ⚠ No models successfully trained")

except Exception as e:
    print(f"\n⚠ Model comparison failed: {e}")
    results_df = pd.DataFrame()
    best_model_name = "None"


### 6.4 Train Stacking Ensemble


In [ ]:
print("=" * 80)
print("6.4 — Training Stacking Ensemble")
print("=" * 80)

stacking_model, stacking_results = train_stacking_regressor(
        X_train_reg, y_train_reg,
        cv=CV_FOLDS,
        ensure_nonnegative=True
        )

print(f"\n✓ Stacking Ensemble Trained:")
print(f"  Base models: {', '.join(stacking_results.get('base_models', []))}")
print(f"  Meta-learner: {stacking_results.get('meta_model', 'Unknown')}")
print(f"  Train R²: {stacking_results.get('train_score', 0):.4f}")
print(f"  CV R² (mean ± std): {stacking_results.get('cv_score', 0):.4f} ± {stacking_results.get('cv_std', 0):.4f}")

# Test set predictions
y_pred_stacking = stacking_model.predict(X_test_reg)

test_metrics = {
    'mae': mean_absolute_error(y_test_reg, y_pred_stacking),
    'rmse': np.sqrt(mean_squared_error(y_test_reg, y_pred_stacking)),
    'r2': r2_score(y_test_reg, y_pred_stacking)
    }

print(f"\n📊 Test Set Performance:")
print(f"  MAE: {test_metrics['mae']:.2f}")
print(f"  RMSE: {test_metrics['rmse']:.2f}")
print(f"  R²: {test_metrics['r2']:.4f}")


### 6.5 Quantile Regression for Prediction Intervals


In [ ]:
print("=" * 80)
print("6.5 — Quantile Regression for Uncertainty Estimation")
print("=" * 80)

quantile_models, quantile_results = train_quantile_regressor(
        X_train_reg, y_train_reg,
        quantiles=QUANTILES
        )

print(f"\n✓ Quantile Models Trained:")
print(f"  Quantiles: {QUANTILES}")
print(f"  Models: {len(quantile_models)}")

# Generate predictions for each quantile
predictions_quantile = {}
for q, model in zip(QUANTILES, quantile_models):
    predictions_quantile[q] = model.predict(X_test_reg)
    try:
        score = model.score(X_train_reg, y_train_reg)
        print(f"  Q{q}: {score:.4f} (train R²)")
    except:
        print(f"  Q{q}: Model trained successfully")


### 6.6 Sector-Specific Models (Optional)


In [ ]:
print("=" * 80)
print("6.6 — Sector-Specific Model Training")
print("=" * 80)

if 'sector' in all_stocks_enhanced.columns:
    feature_cols = list(X_train_reg.columns)

    models, sector_results = train_sector_specific_models(
            all_stocks_enhanced,
            feature_cols=feature_cols,
            target_col=target_col,
            sector_col='sector',
            model_type='random_forest',
            min_samples=MIN_SECTOR_SAMPLES
            )

    print(f"\n✓ Sector-Specific Models Trained:")
    print(f"  Total sectors: {len(models)}")

    sector_metrics = sector_results.get('sector_metrics', sector_results)
    sector_summary = pd.DataFrame(sector_metrics).T

    if 'r2' in sector_summary.columns:
        sector_summary = sector_summary.sort_values('r2', ascending=False)
        print(f"\n📊 Top Sector Model Performance:")
        display_cols = [c for c in ['train_score', 'r2', 'mae', 'rmse']
                        if c in sector_summary.columns]
        if display_cols:
            print(sector_summary[display_cols].head(5).to_string())
else:
    print("\n⚠ Sector column not found - skipping sector-specific models")


### 6.7 Model Persistence


In [ ]:
print("=" * 80)
print("6.7 — Model Persistence")
print("=" * 80)

models_dir = OUTPUT_DIR / 'models'
models_dir.mkdir(exist_ok=True, parents=True)

# Save stacking model
stacking_metadata = {
    'model_type': 'stacking_ensemble',
    'features': list(X_train_reg.columns),
    'target': target_col,
    'date_trained': datetime.now().strftime('%Y-%m-%d'),
    'phase': '9.5',
    'train_score': stacking_results.get('train_score', 0),
    'cv_score': stacking_results.get('cv_score', 0),
    'test_score': test_metrics['r2']
    }

stacking_path = models_dir / 'stacking_ensemble_phase95.joblib'
save_model(stacking_model, str(stacking_path), metadata=stacking_metadata)
print(f"\n✓ Stacking model saved: {stacking_path.name}")

# Save quantile models
for q, model in zip(QUANTILES, quantile_models):
    quantile_metadata = {
        'model_type': f'quantile_regressor_q{q}',
        'features': list(X_train_reg.columns),
        'target': target_col,
        'date_trained': datetime.now().strftime('%Y-%m-%d'),
        'phase': '9.5',
        'quantile': q
        }
    quantile_path = models_dir / f'quantile_q{int(q * 100)}_phase95.joblib'
    save_model(model, str(quantile_path), metadata=quantile_metadata)

print(f"✓ Quantile models saved: {len(QUANTILES)} models")


### 6.8 Summary and Store Predictions


In [ ]:
print("=" * 80)
print("PHASE 9.5 IMPLEMENTATION SUMMARY")
print("=" * 80)

classification_cols = [c for c in all_stocks_with_classification.columns if c.startswith('event_prob_')]

summary = {
    "✓ Classification Features Integrated": f"{len(classification_cols)} probability features + interactions",
    "✓ Models Compared": "6 models: Ridge, Lasso, RF, ET, GB, HistGB",
    "✓ Best Single Model": f"{best_model_name} (R²={results_df.loc[best_model_name, 'r2']:.4f})" if best_model_name != "None" and not results_df.empty else "Not available",
    "✓ Stacking Ensemble": f"R²={test_metrics['r2']:.4f}, MAE={test_metrics['mae']:.2f}",
    "✓ Quantile Regression": f"{len(QUANTILES)} quantiles for prediction intervals",
    "✓ Models Saved": f"{models_dir.name}/ (stacking + quantile models)"
    }

for key, value in summary.items():
    print(f"\n{key}")
    print(f"  {value}")

print(f"\n{'=' * 80}")
print("✓ Phase 9.5 Complete - Advanced Regression System Operational")
print("=" * 80)

# Store predictions in a new dataframe for downstream phases
all_stocks_phase95 = all_stocks_enhanced.copy()
test_indices = X_test_reg.index
valid_indices = test_indices.intersection(all_stocks_phase95.index)

if len(valid_indices) > 0:
    all_stocks_phase95.loc[valid_indices, 'predicted_price_target'] = y_pred_stacking[
        test_indices.isin(valid_indices)
    ]
    all_stocks_phase95.loc[valid_indices, 'prediction_lower_10'] = predictions_quantile[0.1][
        test_indices.isin(valid_indices)
    ]
    all_stocks_phase95.loc[valid_indices, 'prediction_upper_90'] = predictions_quantile[0.9][
        test_indices.isin(valid_indices)
    ]
    print(f"\n✓ Predictions stored in 'all_stocks_phase95': {len(valid_indices):,} samples")

print(f"✓ Dataset ready for Phase 9.6/9.7")


## 7. Model Evaluation and Error Analysis

Comprehensive evaluation including:
- Regression metrics (MAE, RMSE, MAPE, R²)
- Residual analysis
- Sector and region performance breakdown
- SHAP analysis for explainability
- Learning curves and bias-variance diagnosis


In [ ]:
# Comprehensive regression metrics
from finance_ml.eval import comprehensive_regression_metrics, compute_metrics_by_segment

metrics = comprehensive_regression_metrics(y_test_reg, y_pred_stacking)
print("📊 Overall Model Performance:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")


In [ ]:
# Segment analysis (by sector and region)
from finance_ml.eval import compute_metrics_by_segment

# Prepare test data with predictions
test_data = all_stocks_with_classification.loc[X_test_reg.index].copy()
test_data['predicted_price_target'] = y_pred_stacking

sector_metrics = compute_metrics_by_segment(
        test_data, 'price_target', 'predicted_price_target', 'sector'
        )
print("\n📊 Performance by Sector:")
print(sector_metrics)


In [ ]:
# SHAP analysis
from finance_ml.eval import compute_shap_values, create_shap_summary_plot

shap_output_dir = OUTPUT_DIR / "shap"
shap_output_dir.mkdir(exist_ok=True)

create_shap_summary_plot(
        stacking_model,
        X_test_reg,
        output_path=shap_output_dir / "shap_summary.png",
        model_type="tree",
        n_samples=100
        )
print("✓ SHAP analysis complete")


## 8. Identification of Under/Overvalued Stocks with Visualization

Calculate mispricing scores and identify investment opportunities:
- Mispricing score: (Predicted - Current) / Current
- Valuation categories: Severely Undervalued, Undervalued, Fair, Overvalued, Severely Overvalued
- Sector-relative rankings
- Multi-factor scoring (valuation + quality + growth)


In [ ]:
# Calculate mispricing scores
from finance_ml.eval import calculate_mispricing_score, assign_valuation_category

all_stocks_valued = all_stocks_with_classification.copy()
all_stocks_valued['predicted_price_target'] = stacking_model.predict(
        all_stocks_valued[X_train_reg.columns]
        )

all_stocks_valued['mispricing_score'] = calculate_mispricing_score(
        all_stocks_valued,
        predicted_col='predicted_price_target',
        current_col='last_price'
        )

all_stocks_valued['valuation_category'] = assign_valuation_category(
        all_stocks_valued['mispricing_score']
        )

print(f"✓ Valuation Analysis Complete")


In [ ]:
# Rank stocks
from finance_ml.eval import rank_undervalued_stocks, rank_overvalued_stocks

top_undervalued = rank_undervalued_stocks(all_stocks_valued, top_n=20)
top_overvalued = rank_overvalued_stocks(all_stocks_valued, top_n=20)

print("\n🏆 Top 20 Undervalued Stocks (Buy Opportunities):")
print(top_undervalued[['ticker', 'sector', 'mispricing_score', 'valuation_category']].head(20))

print("\n⚠️  Top 20 Overvalued Stocks (Sell Opportunities):")
print(top_overvalued[['ticker', 'sector', 'mispricing_score', 'valuation_category']].head(20))


In [ ]:
# Visualizations
from finance_ml.eval import create_valuation_scatter_plot, create_sector_heatmap

plots_dir = OUTPUT_DIR / "plots"

create_valuation_scatter_plot(
        all_stocks_valued,
        out_path=plots_dir / "valuation_scatter.png",
        color_by='sector'
        )

create_sector_heatmap(
        all_stocks_valued,
        out_path=plots_dir / "sector_heatmap.png",
        metric='mispricing_score'
        )

print("✓ Visualizations created")


## 9. Comprehensive Analytics: Predicted vs. Analyst Price Target Comparison

Compare ML predictions with analyst consensus targets:
- Agreement rate and directional accuracy
- Systematic bias analysis
- Disagreement opportunities (contrarian plays)
- Segment analysis by sector/region
- Calibration and confidence metrics


In [ ]:
# Prediction vs Analyst comparison
from finance_ml.analyst_comparison import PredictionAnalystAnalytics

analytics = PredictionAnalystAnalytics(all_stocks_valued)
analytics.run_full_analysis(
        disagreement_threshold=10.0,
        top_n=50
        )


In [ ]:
# Generate comprehensive Excel report
from finance_ml.eval import generate_prediction_analyst_excel_report

reports_dir = OUTPUT_DIR / "reports"
reports_dir.mkdir(exist_ok=True)

generate_prediction_analyst_excel_report(
        all_stocks_valued,
        excel_path=reports_dir / "prediction_analyst_comparison.xlsx",
        top_n_opportunities=50
        )

print("✓ Excel report generated")


In [ ]:
# Generate PDF report
from finance_ml.eval import generate_enhanced_pdf_report

generate_enhanced_pdf_report(
        all_stocks_valued,
        pdf_path=reports_dir / "stock_valuation_report.pdf",
        title="Stock Price Target Analysis - Comprehensive Report",
        include_financial_dashboard=True,
        include_quality_alerts=True,
        include_hypothesis_tests=True,
        include_charts=False
        )

print("✓ PDF report generated")


## 10. Portfolio Optimization with Risk Metrics

Construct optimized portfolios based on predictions:
- Maximum Sharpe ratio optimization
- Minimum volatility optimization
- Target return optimization
- Risk metrics (VaR, CVaR, Sharpe, Sortino, Max Drawdown)


In [ ]:
# Prepare portfolio data (top undervalued stocks)
from finance_ml.portfolio_optimization import (
    optimize_portfolio_max_sharpe,
    optimize_portfolio_min_volatility,
    generate_efficient_frontier
    )

top_candidates = rank_undervalued_stocks(all_stocks_valued, top_n=50)

# Calculate expected returns (mispricing as proxy)
expected_returns = top_candidates['mispricing_score'].values / 100

# Estimate covariance (simplified - use historical returns in production)
n_stocks = len(top_candidates)
cov_matrix = np.eye(n_stocks) * 0.04  # Simplified example


In [ ]:
# Optimize for maximum Sharpe ratio
optimal_portfolio = optimize_portfolio_max_sharpe(
        expected_returns,
        cov_matrix,
        risk_free_rate=0.02,
        allow_short=False,
        max_weight=0.15
        )

print("✓ Portfolio Optimization Complete")
print(f"  Expected Return: {optimal_portfolio['portfolio_return']:.2%}")
print(f"  Portfolio Volatility: {optimal_portfolio['portfolio_volatility']:.2%}")
print(f"  Sharpe Ratio: {optimal_portfolio['sharpe_ratio']:.3f}")


In [ ]:
# Calculate risk metrics
from finance_ml.risk_metrics import calculate_portfolio_risk_metrics

# Simulated portfolio returns (use actual historical data in production)
portfolio_returns = np.random.normal(0.08 / 252, 0.15 / np.sqrt(252), 252)

risk_metrics_result = calculate_portfolio_risk_metrics(
        pd.Series(portfolio_returns),
        risk_free_rate=0.02,
        confidence_levels=[0.95, 0.99]
        )

print("\n📊 Portfolio Risk Metrics:")
for metric, value in risk_metrics_result.items():
    print(f"  {metric}: {value}")

print("\n✅ Portfolio Optimization Complete")
print("\n" + "=" * 80)
print("WORKFLOW COMPLETE - All 10 sections executed successfully")
print("=" * 80)
